# GeoMERIT demonstration

Geophysical Method Evaluation and Ranking by Investigation Target.

This notebook demonstrates typical use of GeoMERIT: ranking near-surface geophysical methods for a stated investigation target, inspecting the knowledge base, and propagating input uncertainty into a probabilistic recommendation. It reproduces the worked example shown in the manuscript.

## 1. Load the tool

GeoMERIT is a single deterministic class with three inspectable knowledge structures: the method-capability table, the method-by-target contrast matrix, and the criterion weights.

In [ ]:
from geophysical_method_selector import GeophysicalMethodSelector

sel = GeophysicalMethodSelector()
print('Methods:', list(sel.methods.keys()))
print('Target classes:', list(sel.target_contrast.keys()))
print('Weights:', sel.weights)

Methods: ['ERT', 'Induced_Polarization', 'Self_Potential', 'GPR', 'TEM', 'Seismic_Refraction', 'Magnetometry', 'Gravity', 'Radiometric']
Target classes: ['groundwater', 'void', 'archaeology']
Weights: {'physical_contrast': 0.4, 'data_quality': 0.3, 'cost': 0.2, 'effort': 0.1}


## 2. Rank methods for a scenario

A scenario is a small dictionary of named inputs. Here: a saline (conductive) coastal groundwater target at 40 m depth, low noise, moderate budget and schedule, and a moderately high resolution requirement.

In [ ]:
scenario = dict(
    target='groundwater',
    target_depth=40,
    conductivity=100,   # mS/cm, high = saline
    noise_level=20,     # per cent
    budget=5000,        # USD/km
    time_constraint=3,  # days/km
    required_resolution=0.7,
)

ranked = sel.rank_methods(scenario)
for method, score in ranked:
    print(f'{method:22s} {score:6.2f}')

ERT                     77.15
TEM                     58.68
Induced_Polarization    57.54
Self_Potential          48.97
Seismic_Refraction      48.50
Magnetometry            28.76
GPR                     26.03
Gravity                 14.22
Radiometric             12.88


For this conductive groundwater target, ERT leads, followed by the electromagnetic and galvanic methods (TEM, IP, SP). GPR ranks low because its signal is attenuated in conductive ground. This is the behaviour reported as the worked example in the manuscript.

## 3. Inspect the knowledge base

Every recommendation traces to the published tables. Any value can be edited and the effect seen immediately. View the contrast values for the groundwater class:

In [ ]:
for m in sel.methods:
    print(f'{m:22s} X(m, groundwater) = ' + format(sel.target_contrast['groundwater'][m], '.2f'))

ERT                    X(m, groundwater) = 0.95
Induced_Polarization   X(m, groundwater) = 0.75
Self_Potential         X(m, groundwater) = 0.55
GPR                    X(m, groundwater) = 0.30
TEM                    X(m, groundwater) = 0.90
Seismic_Refraction     X(m, groundwater) = 0.70
Magnetometry           X(m, groundwater) = 0.15
Gravity                X(m, groundwater) = 0.20
Radiometric            X(m, groundwater) = 0.05


## 4. Compare target classes

The same site conditions give different rankings for different targets, because the physics of detection differs. Compare the top methods for a shallow archaeological target.

In [ ]:
arch = dict(target='archaeology', target_depth=2, conductivity=20,
            noise_level=20, budget=5000, time_constraint=3,
            required_resolution=0.9)
for method, score in sel.rank_methods(arch)[:4]:
    print(f'{method:22s} {score:6.2f}')

GPR                     78.26
Magnetometry            73.78
ERT                     60.85
Self_Potential          38.45


## 5. Probabilistic recommendation under input uncertainty

Input parameters are uncertain. Propagating that uncertainty by Monte Carlo gives the probability that each method occupies first place, turning the deterministic ranking into a probabilistic recommendation.

In [ ]:
import numpy as np

rng = np.random.default_rng(20260701)
N = 2000
COND = {20:(5,35), 50:(30,80), 100:(70,160)}

def perturb(s):
    lo, hi = COND.get(int(s['conductivity']), (s['conductivity']*0.6, s['conductivity']*1.4))
    return dict(target=s['target'],
                target_depth=s['target_depth']*rng.uniform(0.6,1.4),
                conductivity=rng.uniform(lo,hi),
                noise_level=min(95,max(0,s['noise_level']+rng.uniform(-15,15))),
                budget=s['budget']*rng.uniform(0.7,1.6),
                time_constraint=s['time_constraint']*rng.uniform(0.7,1.6),
                required_resolution=s['required_resolution'])

counts = {m:0 for m in sel.methods}
for _ in range(N):
    top = sel.rank_methods(perturb(scenario))[0][0]
    counts[top] += 1

for m, c in sorted(counts.items(), key=lambda x:-x[1]):
    if c:
        print(f'{m:22s} P(rank 1) = ' + format(c/N, '.3f'))

ERT                    P(rank 1) = 1.000


For this scenario ERT occupies first place in every draw, so the recommendation is completely stable to the stated input uncertainty.

## 6. Reproduce the full study

The complete literature benchmark, robustness, weighting, uncertainty, and failure analyses are regenerated by:

```
python reproduce.py
```

The machine-readable benchmark is in `benchmark_cases.csv`, and every result and figure traces to documented parameters.